# Plan B — Resultados finales: HMM-M vs 4 baselines SOTA

Agrega los resultados de Plan B (HMM+TransformerCommon en M, channel-independence) y los baselines TSLib corridos con sus scripts originales en M.

**Baselines:** PatchTST (Nie et al., 2023), DLinear (Zeng et al., 2023), TimeMixer (Wang et al., 2024b), TimeXer (Wang et al., 2024a).

**Datasets Grupo 1:** ETTh1, ETTh2, Weather, Electricity. **Horizontes:** {96, 192, 336, 720}.

**Métrica:** MSE/MAE promediadas sobre todos los canales y muestras (consistente con leaderboard TSLib).

Requiere haber completado:
1. `scripts/plan_b/train_hmm_caches.py` (caches HMM-M).
2. `scripts/plan_b/run_ritmo_sweep.py` (K sweep RITMO).
3. Los scripts TSLib en `scripts/long_term_forecast/`.

In [ ]:
# Celda 1. Imports, workdir y config
import os, sys, re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = '/home/jaime/TFG/RITMO'
os.chdir(REPO)
if REPO not in sys.path: sys.path.insert(0, REPO)

plt.style.use('seaborn-v0_8-paper')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['savefig.bbox'] = 'tight'
COLORS_OI = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', '#000000']

RESULTS_DIR = Path('results')
FIG_DIR = Path('notebooks/figures_plan_b'); FIG_DIR.mkdir(exist_ok=True, parents=True)

# Datasets canonicos (mismo naming que scripts/plan_b/plan_b_config.py)
DATASET_MAP = {
    'ETTh1':       {'model_id_prefix': 'ETTh1'},
    'ETTh2':       {'model_id_prefix': 'ETTh2'},
    'Weather':     {'model_id_prefix': 'weather'},
    'Electricity': {'model_id_prefix': 'ECL'},
}
HORIZONS = [96, 192, 336, 720]
BASELINES = ['PatchTST', 'DLinear', 'TimeMixer', 'TimeXer']
print('Setup OK')

In [ ]:
# Celda 2. Cargar resultados RITMO-M desde results/plan_b_PLANB_*_0/metrics.npy
ritmo_rows = []
for p in sorted(RESULTS_DIR.glob('plan_b_PLANB_*_0')):
    m = p / 'metrics.npy'
    if not m.exists(): continue
    mae, mse, rmse, mape, mspe = np.load(m)
    m_tag = re.search(r'PLANB_([^_]+)_(hmm_soft_residual|hmm_soft|hmm_augmented|hmm_split|hmm_patched|hmm)_K(\d+)_pl(\d+)', p.name)
    if not m_tag:
        print(f'skip (tag): {p.name}'); continue
    ds, variant, K, pl = m_tag.group(1), m_tag.group(2), int(m_tag.group(3)), int(m_tag.group(4))
    ritmo_rows.append({'dataset': ds, 'variant': variant, 'K': K, 'pred_len': pl,
                      'MSE': float(mse), 'MAE': float(mae)})
ritmo = pd.DataFrame(ritmo_rows)
print(f'RITMO-M runs cargados: {len(ritmo)}')
if len(ritmo) == 0:
    raise SystemExit('Sin runs RITMO-M. Ejecutar scripts/plan_b/run_ritmo_sweep.py primero.')

# Mejor (variant, K) por dataset (min MSE promedio sobre horizontes disponibles).
ritmo_avg = ritmo.groupby(['dataset', 'variant', 'K'])['MSE'].mean().reset_index(name='MSE_avg')
best = ritmo_avg.loc[ritmo_avg.groupby('dataset')['MSE_avg'].idxmin()].reset_index(drop=True)
print('\nConfiguracion ganadora RITMO-M por dataset:')
print(best.to_string(index=False))

ritmo_best = ritmo.merge(best[['dataset','variant','K']], on=['dataset','variant','K'], how='inner')
ritmo_best = ritmo_best.sort_values(['dataset','pred_len']).reset_index(drop=True)
print('\nFilas RITMO-M ganadoras por horizonte:')
print(ritmo_best[['dataset','variant','K','pred_len','MSE','MAE']].to_string(index=False))

In [ ]:
# Celda 3. Cargar resultados de los 4 baselines SOTA (scripts long_term_forecast).
def _parse_ltf_dir(p):
    name = p.name
    if not name.startswith('long_term_forecast_'): return None
    if '_ftM_' not in name: return None
    m = re.search(r'_pl(\d+)_', name)
    if not m: return None
    pl = int(m.group(1))
    if pl not in HORIZONS: return None
    model = None
    for b in BASELINES:
        if f'_{b}_' in name:
            model = b; break
    if model is None: return None
    ds = None
    for ds_name, info in DATASET_MAP.items():
        if f"long_term_forecast_{info['model_id_prefix']}_" in name:
            ds = ds_name; break
    if ds is None: return None
    return ds, model, pl

baseline_rows = []
for p in RESULTS_DIR.iterdir():
    if not p.is_dir(): continue
    parsed = _parse_ltf_dir(p)
    if parsed is None: continue
    ds, model, pl = parsed
    mpath = p / 'metrics.npy'
    if not mpath.exists(): continue
    mae, mse, rmse, mape, mspe = np.load(mpath)
    baseline_rows.append({'dataset': ds, 'model': model, 'pred_len': pl,
                          'MSE': float(mse), 'MAE': float(mae), 'dir': p.name})
bdf = pd.DataFrame(baseline_rows)
print(f'Baseline runs encontrados: {len(bdf)}')
if len(bdf):
    bdf = bdf.sort_values('MSE').drop_duplicates(['dataset','model','pred_len'], keep='first')
    print(bdf.pivot_table(index=['dataset','pred_len'], columns='model', values='MSE').to_string())
else:
    print('AVISO: sin baselines todavia. Correr los scripts de scripts/long_term_forecast/.')

In [ ]:
# Celda 4. Construir tablas de comparacion (una por dataset): RITMO + 4 baselines.
def _build_dataset_table(ds_name):
    cols = ['RITMO-M'] + BASELINES
    table_mse = pd.DataFrame(index=HORIZONS, columns=cols, dtype=float)
    table_mae = pd.DataFrame(index=HORIZONS, columns=cols, dtype=float)
    for _, r in ritmo_best[ritmo_best['dataset']==ds_name].iterrows():
        table_mse.loc[r['pred_len'], 'RITMO-M'] = r['MSE']
        table_mae.loc[r['pred_len'], 'RITMO-M'] = r['MAE']
    for model in BASELINES:
        sub = bdf[(bdf['dataset']==ds_name) & (bdf['model']==model)] if len(bdf) else pd.DataFrame()
        for _, r in sub.iterrows():
            table_mse.loc[r['pred_len'], model] = r['MSE']
            table_mae.loc[r['pred_len'], model] = r['MAE']
    return table_mse, table_mae

tables_mse, tables_mae = {}, {}
for ds_name in DATASET_MAP:
    tm, ta = _build_dataset_table(ds_name)
    tables_mse[ds_name] = tm; tables_mae[ds_name] = ta
    print(f"\n=== {ds_name} ===")
    print('MSE:'); print(tm.round(4).to_string())
    print('MAE:'); print(ta.round(4).to_string())

In [ ]:
# Celda 5. Tabla agregada sobre horizontes + ganador por dataset.
agg_rows = []
for ds_name in DATASET_MAP:
    tm = tables_mse[ds_name]; ta = tables_mae[ds_name]
    for col in tm.columns:
        agg_rows.append({
            'dataset': ds_name, 'model': col,
            'MSE_avg': float(tm[col].mean(skipna=True)) if tm[col].notna().any() else np.nan,
            'MAE_avg': float(ta[col].mean(skipna=True)) if ta[col].notna().any() else np.nan,
            'coverage': int(tm[col].notna().sum()),
        })
agg = pd.DataFrame(agg_rows)
pivot_mse = agg.pivot(index='dataset', columns='model', values='MSE_avg').round(4)
pivot_mae = agg.pivot(index='dataset', columns='model', values='MAE_avg').round(4)
print('\nMSE promedio:'); print(pivot_mse.to_string())
print('\nMAE promedio:'); print(pivot_mae.to_string())
print('\nGanador por dataset (MSE avg):')
winners = pivot_mse.idxmin(axis=1).to_frame('winner')
winners['MSE_win'] = pivot_mse.min(axis=1)
print(winners.to_string())

pivot_mse.to_csv(RESULTS_DIR / 'plan_b_table_mse.csv')
pivot_mae.to_csv(RESULTS_DIR / 'plan_b_table_mae.csv')
agg.to_csv(RESULTS_DIR / 'plan_b_agg.csv', index=False)
print(f"\nGuardado CSVs en {RESULTS_DIR}")

In [ ]:
# Celda 6. Barplots MSE por dataset.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, ds_name in zip(axes.flat, DATASET_MAP):
    tm = tables_mse[ds_name]
    tm.plot(kind='bar', ax=ax, width=0.8, color=COLORS_OI[:len(tm.columns)], edgecolor='black', linewidth=0.3)
    ax.set_title(ds_name, fontweight='bold')
    ax.set_xlabel('pred_len'); ax.set_ylabel('MSE')
    ax.grid(alpha=0.3, axis='y')
    ax.legend(loc='best', fontsize=8)
    ax.tick_params(axis='x', rotation=0)
plt.suptitle('Plan B: HMM-M vs 4 baselines SOTA (features M, leaderboard TSLib)', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'plan_b_barplot_mse.png'); plt.savefig(FIG_DIR / 'plan_b_barplot_mse.pdf')
plt.show()

In [ ]:
# Celda 7. Heatmap ranking (1 = mejor por dataset).
ranked = pivot_mse.rank(axis=1, method='min')
fig, ax = plt.subplots(figsize=(7, 3.5))
im = ax.imshow(ranked.values, cmap='RdYlGn_r', aspect='auto', vmin=1, vmax=len(ranked.columns))
ax.set_xticks(range(len(ranked.columns))); ax.set_xticklabels(ranked.columns, rotation=30, ha='right')
ax.set_yticks(range(len(ranked.index))); ax.set_yticklabels(ranked.index)
for i, row in enumerate(ranked.values):
    for j, v in enumerate(row):
        if np.isnan(v): continue
        ax.text(j, i, f'{int(v)}', ha='center', va='center',
                color='white' if v >= 3 else 'black', fontweight='bold')
ax.set_title('Ranking por MSE promedio (1 = mejor)', fontweight='bold')
plt.colorbar(im, ax=ax, label='ranking')
plt.tight_layout()
plt.savefig(FIG_DIR / 'plan_b_ranking_heatmap.png'); plt.savefig(FIG_DIR / 'plan_b_ranking_heatmap.pdf')
plt.show()

In [ ]:
# Celda 8. Tabla paper-style por dataset (MSE + MAE + Avg).
def _paper_style_table(ds_name):
    tm = tables_mse[ds_name]; ta = tables_mae[ds_name]
    rows = []
    for pl in HORIZONS:
        row_mse = {'dataset': ds_name, 'pred_len': pl, 'metric': 'MSE'}
        row_mae = {'dataset': ds_name, 'pred_len': pl, 'metric': 'MAE'}
        for col in tm.columns:
            row_mse[col] = tm.loc[pl, col]; row_mae[col] = ta.loc[pl, col]
        rows += [row_mse, row_mae]
    row_avg_mse = {'dataset': ds_name, 'pred_len': 'Avg', 'metric': 'MSE'}
    row_avg_mae = {'dataset': ds_name, 'pred_len': 'Avg', 'metric': 'MAE'}
    for col in tm.columns:
        row_avg_mse[col] = tm[col].mean(skipna=True); row_avg_mae[col] = ta[col].mean(skipna=True)
    rows += [row_avg_mse, row_avg_mae]
    return pd.DataFrame(rows)

paper_tables = {ds: _paper_style_table(ds) for ds in DATASET_MAP}
all_paper = pd.concat(paper_tables.values(), ignore_index=True)
all_paper.to_csv(RESULTS_DIR / 'plan_b_paper_table.csv', index=False)
for ds, tbl in paper_tables.items():
    print(f"\n=== {ds} ===")
    print(tbl.round(4).to_string(index=False))
print(f"\nGuardado: {RESULTS_DIR/'plan_b_paper_table.csv'}")